In [0]:
# Esta tabla tendrá una fila por año y una columna para cada indicador.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Leer datos limpios
df = spark.table(
    "world_bank.costa_rica.silver_economic_indicators"
)

gold_summary = (
    df.groupBy("year")
    .pivot("indicator_code", [
        "NY.GDP.MKTP.KD.ZG",
        "FP.CPI.TOTL.ZG",
        "SL.UEM.TOTL.ZS",
        "SP.POP.TOTL"
    ])
    .agg(F.first("value"))
    .select(
        "year",
        F.col("`NY.GDP.MKTP.KD.ZG`").alias("gdp_growth_pct"),
        F.col("`FP.CPI.TOTL.ZG`").alias("inflation_pct"),
        F.col("`SL.UEM.TOTL.ZS`").alias("unemployment_pct"),
        F.col("`SP.POP.TOTL`").alias("population")
    )
)

gold_summary.write.format("delta").mode("overwrite").saveAsTable(
    "world_bank.costa_rica.gold_economic_summary"
)

display(gold_summary.orderBy(F.desc("year")))